# Critical Points Analysis

Analysis of physics simulation data with columns: **phi1, phi2, zeta, R, E**

**Finds:**
1. Configuration with global minimum R
2. Configuration with maximum R among minimum R values  
3. Custom area searches

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
INTERACTION_TYPE = "DD"
data = np.loadtxt(os.path.join(f"../E_all_{INTERACTION_TYPE}.dat"))

In [ ]:
class DataAnalyzer:
    """
    A class to analyze physics simulation data with columns: phi1, phi2, zeta, R, E
    """
    
    def __init__(self, data):
        """
        Initialize with data array
        Args:
            data: numpy array with columns [phi1, phi2, zeta, R, E]
        """
        self.data = data
        self.columns = ['phi1', 'phi2', 'zeta', 'R', 'E']
        self._grouped_data = None
        
    def _group_by_config(self):
        """Group data by configuration (phi1, phi2, zeta)"""
        if self._grouped_data is None:
            self._grouped_data = {}
            for i, row in enumerate(self.data):
                config = (row[0], row[1], row[2])  # phi1, phi2, zeta
                if config not in self._grouped_data:
                    self._grouped_data[config] = []
                self._grouped_data[config].append((i, row))
        return self._grouped_data
    
    def find_min_R_configurations(self):
        """
        Find configurations with minimum R for each (phi1, phi2, zeta) group
        Returns: list of tuples (config, min_R_row_index, min_R_row_data)
        """
        grouped = self._group_by_config()
        min_R_configs = []
        
        for config, rows in grouped.items():
            # Find minimum R in this configuration
            min_R_idx = min(rows, key=lambda x: x[1][3])  # index 3 is R column
            min_R_configs.append((config, min_R_idx[0], min_R_idx[1]))
            
        return min_R_configs
    
    def find_global_min_R_config(self):
        """
        Find the configuration with the absolute minimum R
        Returns: (config, row_index, row_data)
        """
        min_R_configs = self.find_min_R_configurations()
        global_min = min(min_R_configs, key=lambda x: x[2][3])  # index 3 is R column
        return global_min
    
    def find_max_R_min_config(self):
        """
        Find the configuration with maximum R among all minimum R values
        Returns: (config, row_index, row_data)
        """
        min_R_configs = self.find_min_R_configurations()
        max_R_min = max(min_R_configs, key=lambda x: x[2][3])  # index 3 is R column
        return max_R_min
    
    def find_configs_in_area(self, E_range=None, R_range=None, max_results=10):
        """
        Find configurations within specified E and R ranges
        Args:
            E_range: tuple (min_E, max_E) or None
            R_range: tuple (min_R, max_R) or None
            max_results: maximum number of results to return
        Returns: list of tuples (config, row_index, row_data)
        """
        results = []
        
        for i, row in enumerate(self.data):
            # Check E range
            if E_range and not (E_range[0] <= row[4] <= E_range[1]):
                continue
            # Check R range    
            if R_range and not (R_range[0] <= row[3] <= R_range[1]):
                continue
                
            config = (row[0], row[1], row[2])
            results.append((config, i, row))
            
            if len(results) >= max_results:
                break
                
        return results
    
    def print_result(self, result, title="Result"):
        """Pretty print a single result"""
        if isinstance(result, tuple) and len(result) == 3:
            config, idx, row = result
            print(f"\n{title}:")
            print(f"  Configuration: phi1={config[0]:.6f}, phi2={config[1]:.6f}, zeta={config[2]:.6f}")
            print(f"  Row index: {idx}")
            print(f"  Values: R={row[3]:.6f}, E={row[4]:.6f}")
        else:
            print(f"\n{title}: {result}")
    
    def print_results(self, results, title="Results"):
        """Pretty print multiple results"""
        print(f"\n{title} ({len(results)} items):")
        for i, result in enumerate(results):
            config, idx, row = result
            print(f"  {i+1}. phi1={config[0]:.6f}, phi2={config[1]:.6f}, zeta={config[2]:.6f}, "
                  f"R={row[3]:.6f}, E={row[4]:.6f} (row {idx})")
    
    def get_summary_stats(self):
        """Get summary statistics of the data"""
        stats = {}
        for i, col in enumerate(self.columns):
            stats[col] = {
                'min': self.data[:, i].min(),
                'max': self.data[:, i].max(),
                'mean': self.data[:, i].mean(),
                'std': self.data[:, i].std()
            }
        return stats

In [ ]:
# Create analyzer and find critical points
analyzer = DataAnalyzer(data)

print("CRITICAL POINTS ANALYSIS")
print("="*50)

# 1. Global minimum R configuration
print("\n1. GLOBAL MINIMUM R:")
global_min_R = analyzer.find_global_min_R_config()
config, idx, row = global_min_R
print(f"   phi1={config[0]:.1f}, phi2={config[1]:.1f}, zeta={config[2]:.6f}")
print(f"   R={row[3]:.3f}, E={row[4]:.3f}")

# 2. Maximum R among minimum R values  
print("\n2. MAXIMUM R (among min R values):")
max_R_min = analyzer.find_max_R_min_config()
config, idx, row = max_R_min
print(f"   phi1={config[0]:.1f}, phi2={config[1]:.1f}, zeta={config[2]:.6f}")
print(f"   R={row[3]:.3f}, E={row[4]:.3f}")

print("\n" + "="*50)

In [ ]:
# Interactive search functions
def search_area(R_min=None, R_max=None, E_min=None, E_max=None, max_results=10):
    """Search for configurations in specified R and/or E ranges"""
    R_range = (R_min, R_max) if R_min is not None and R_max is not None else None
    E_range = (E_min, E_max) if E_min is not None and E_max is not None else None
    
    results = analyzer.find_configs_in_area(E_range=E_range, R_range=R_range, max_results=max_results)
    
    print(f"\nFound {len(results)} configurations:")
    for i, (config, idx, row) in enumerate(results):
        print(f"{i+1}. phi1={config[0]:.1f}, phi2={config[1]:.1f}, zeta={config[2]:.6f}, "
              f"R={row[3]:.3f}, E={row[4]:.3f}")
    return results

def get_config_data(phi1, phi2, zeta):
    """Get all R,E values for a specific configuration"""
    indices = np.where((data[:, 0] == phi1) & (data[:, 1] == phi2) & (data[:, 2] == zeta))[0]
    
    if len(indices) == 0:
        print(f"No data found for phi1={phi1}, phi2={phi2}, zeta={zeta}")
        return []
    
    config_data = data[indices]
    print(f"\nConfiguration: phi1={phi1}, phi2={phi2}, zeta={zeta}")
    print(f"R values: {config_data[:,3].min():.3f} to {config_data[:,3].max():.3f}")
    print(f"E values: {config_data[:,4].min():.3f} to {config_data[:,4].max():.3f}")
    return config_data

In [ ]:
# Usage examples
print("\nUSAGE EXAMPLES:")
print("search_area(R_min=8.8, R_max=8.9)")
print("search_area(E_min=-12440, E_max=-12435)")  
print("get_config_data(phi1=22.0, phi2=19.0, zeta=0.465625)")

print(f"\nData: {len(data):,} points, R=[{data[:,3].min():.1f},{data[:,3].max():.1f}], E=[{data[:,4].min():.0f},{data[:,4].max():.0f}]")